# CMPT 413 / 713 Tutorial 3
## Text Classification with PyTorch

In [2]:
import torch
import torchtext

# load the AG NEWS dataset from the torchtext library

train_dataset, val_dataset = torchtext.datasets.AG_NEWS()
target_classes = ["World", "Sports", "Business", "Sci/Tec"]

In [3]:
# visualize the data in train split

tmp = iter(train_dataset)

print(f"First data: {next(tmp)}\n")
print(f"Second data: {next(tmp)}\n")
print(f"Third data: {next(tmp)}\n")

First data: (3, "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")

Second data: (3, 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.')

Third data: (3, "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.")



In [5]:
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

# use a simple tokenizer to separate words and punctuation marks

tokenizer = get_tokenizer("basic_english")

def build_vocab(datasets):
    for dataset in datasets:
        for _, text in dataset:
            yield tokenizer(text)

vocab = build_vocab_from_iterator(build_vocab([train_dataset, val_dataset]), specials=["<UNK>"])
vocab.set_default_index(vocab["<UNK>"])

In [6]:
# visualize the tonenized data

print(len(vocab.get_itos()))
print(vocab.get_itos())

98635
['<UNK>', '.', 'the', ',', 'to', 'a', 'of', 'in', 'and', 's', 'on', 'for', '#39', '(', ')', '-', "'", 'that', 'with', 'as', 'at', 'is', 'its', 'new', 'by', 'it', 'said', 'reuters', 'has', 'from', 'an', 'ap', 'his', 'will', 'after', 'was', 'us', 'be', 'over', 'have', 'their', '&lt', 'are', 'up', 'quot', 'first', 'but', 'more', 'two', 'he', 'world', 'u', 'this', '--', 'company', 'monday', 'wednesday', 'tuesday', 'out', 'thursday', 'oil', 'one', 'not', 'against', 'inc', 'friday', 'into', 'they', 'about', 'last', 'iraq', 'year', 'than', 'york', 'who', 'yesterday', 'microsoft', 'president', 'were', 'no', '?', 'been', 'million', 't', 'says', 'week', 'had', 'corp', 'united', 'game', 'when', 'sunday', 'prices', 'could', 'three', 'would', 'years', 'group', 'government', 'time', 'today', 'security', 'people', 'afp', 'may', 'which', 'percent', 'software', '1', 'win', 'next', 'off', 'team', 'back', 'saturday', 'night', 'or', 'china', 'internet', 'season', '2', 'deal', 'some', 'can', 'sales',

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from torch.utils.data import DataLoader
from torchtext.data.functional import to_map_style_dataset

# set-up data loaders (vectorize text data)

vectorizer = CountVectorizer(vocabulary=vocab.get_itos(), tokenizer=tokenizer)

def vectorize_batch(batch):
    y, x = list(zip(*batch))
    x = vectorizer.transform(x).todense()
    return torch.tensor(x, dtype=torch.float32, device="cuda"), torch.tensor(y, device="cuda") - 1  # we have deducted 1 from target names to get them in range [0,1,2,3] from [1,2,3,4]

train_dataset, val_dataset = to_map_style_dataset(train_dataset), to_map_style_dataset(val_dataset)
train_loader = DataLoader(train_dataset, batch_size=1024, collate_fn=vectorize_batch)
val_loader  = DataLoader(val_dataset, batch_size=1024, collate_fn=vectorize_batch)

In [8]:
# visualize the train split data loadter

for x, y in train_loader:
    print(x.shape, y.shape)
    print(x[0])
    print(y[0])
    break

torch.Size([1024, 98635]) torch.Size([1024])
tensor([0., 2., 1.,  ..., 0., 0., 0.], device='cuda:0')
tensor(2, device='cuda:0')


In [23]:
# define the network

from torch import nn
from torch.nn import functional as F

# single layer
# class TextClassifier(nn.Module):
#     def __init__(self):
#         super(TextClassifier, self).__init__()
#         self.layer = nn.Linear(len(vocab), 4)

#     def forward(self, x_batch):
#         return self.layer(x_batch)

# 3-layer
class TextClassifier(nn.Module):
    def __init__(self):
        super(TextClassifier, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(len(vocab), 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x_batch):
        return self.layers(x_batch)

In [24]:
# visualize network's output

text_classifier_model = TextClassifier()
text_classifier_model.to("cuda") # move model to GPU for acceleration

for x, y in train_loader:
    y_preds = text_classifier_model(x)
    print(y_preds.shape)
    print(y_preds)
    break

torch.Size([1024, 4])
tensor([[ 0.1121,  0.1005, -0.0499, -0.0513],
        [ 0.1153,  0.0982, -0.0481, -0.0498],
        [ 0.1159,  0.0996, -0.0458, -0.0494],
        ...,
        [ 0.1155,  0.0991, -0.0520, -0.0485],
        [ 0.1133,  0.0959, -0.0493, -0.0499],
        [ 0.1133,  0.0995, -0.0499, -0.0488]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


In [25]:
from sklearn.metrics import accuracy_score

# define the evaluation code

def cal_loss_and_accuracy(model, loss_fn, val_data_loader):
    with torch.no_grad():
        y_actual, y_preds, losses = [],[],[]
        for x, y in val_data_loader:
            preds = model(x)
            loss = loss_fn(preds, y)
            losses.append(loss.item())
            y_actual.append(y)
            y_preds.append(preds.argmax(dim=-1))

        y_actual = torch.cat(y_actual)
        y_preds = torch.cat(y_preds)

        print("Val Loss : {:.3f}".format(torch.tensor(losses).mean()))
        print("Val Accuracy  : {:.3f}".format(accuracy_score(y_actual.cpu().numpy(), y_preds.cpu().numpy())))

## Training code

In [26]:
from torch.optim import Adam
from tqdm import tqdm

loss_fn = nn.CrossEntropyLoss() # loss function, it includes the softmax operation
optimizer = Adam(text_classifier_model.parameters(), lr=0.0001) # optimizer

# train the model 5 epochs
for i in range(5):
    losses = []
    for x, y in tqdm(train_loader):
        
        # forward
        y_preds = text_classifier_model(x)
        
        # calculate losses
        loss = loss_fn(y_preds, y)
        losses.append(loss.item())
        
        # empty old gradients
        optimizer.zero_grad()
        
        # calculate gradients and backward
        loss.backward()
        
        # update all parameters
        optimizer.step()

    print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
    cal_loss_and_accuracy(text_classifier_model, loss_fn, val_loader)

100%|█████████████████████████████████████████| 118/118 [00:25<00:00,  4.65it/s]


Train Loss : 1.262
Val Loss : 1.042
Val Accuracy  : 0.856


100%|█████████████████████████████████████████| 118/118 [00:25<00:00,  4.71it/s]


Train Loss : 0.769
Val Loss : 0.548
Val Accuracy  : 0.887


100%|█████████████████████████████████████████| 118/118 [00:24<00:00,  4.80it/s]


Train Loss : 0.428
Val Loss : 0.373
Val Accuracy  : 0.900


100%|█████████████████████████████████████████| 118/118 [00:24<00:00,  4.74it/s]


Train Loss : 0.312
Val Loss : 0.314
Val Accuracy  : 0.906


100%|█████████████████████████████████████████| 118/118 [00:25<00:00,  4.71it/s]


Train Loss : 0.259
Val Loss : 0.285
Val Accuracy  : 0.911


## Inference code

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

y_actual, y_preds = [], []

# test the model
with torch.no_grad():
    for x, y in val_loader:
        preds = text_classifier_model(x)
        y_preds.append(preds)
        y_actual.append(y)
    
    y_preds, y_actual = torch.cat(y_preds), torch.cat(y_actual)

    y_actual = y_actual.cpu().numpy()
    y_preds = F.softmax(y_preds, dim=-1).argmax(dim=-1).cpu().numpy()

# print results
print("Accuracy : {}".format(accuracy_score(y_actual, y_preds)))
print("\nClassification Report: ")
print(classification_report(y_actual, y_preds, target_names=target_classes))

Code is modified from https://coderzcolumn.com/tutorials/artificial-intelligence/pytorch-simple-guide-to-text-classification